In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Unzipping Dataset from Google Drive in Colab

In [ ]:
!unzip -o "/content/drive/MyDrive/nutrivision_final3.zip" -d "/content/"

Streaming output truncated to the last 5000 lines.
  inflating: /content/nutrivision_merged/valid/labels/ds1_images-76-_jpeg.rf.7b9e618741629e65221662a51be459de.txt  
  inflating: /content/nutrivision_merged/valid/labels/ds3_maxresdefault-8-_jpg.rf.c7e076eaf03ae9a96c7556154db5d2eb.txt  
  inflating: /content/nutrivision_merged/valid/labels/ds1_images-2024-10-11T223221-915_jpeg.rf.356f3da067457522f8db72d5bd49db43.txt  
  inflating: /content/nutrivision_merged/valid/labels/ds1_images-208-_jpeg.rf.e3dab3a82919d4695199d83681d09597.txt  
  inflating: /content/nutrivision_merged/valid/labels/ds1_images-2024-10-25T033423-365_jpg.rf.334894969d9d58f91c83144fca2f43c5.txt  
  inflating: /content/nutrivision_merged/valid/labels/ds1_images-75-_jpeg.rf.60fce1efa22b169fc66d7c4e93df44b6.txt  
  inflating: /content/nutrivision_merged/valid/labels/ds1_Screenshot-2024-10-24-165131_png.rf.dcef15877222988791246c886eb9c0a8.txt  
  inflating: /content/nutrivision_merged/valid/labels/ds1_Header-pidi_jpeg.rf.5

# Install Dependencies

In [ ]:
!pip install torch torchvision opencv-python

# Import Libraries

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os
import cv2
import numpy as np
from torchvision import transforms, models

device = torch.device("cpu")
print(device)

cpu


# Image Dataset Class

In [ ]:
class YOLODataset(Dataset):
    def __init__(self, img_dir, label_dir, transform=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transform = transform

        self.images = []

        # ✅ Only keep images with valid labels
        for img_name in os.listdir(img_dir):
            label_path = os.path.join(
                label_dir,
                img_name.replace(".jpg", ".txt").replace(".png", ".txt")
            )

            if not os.path.exists(label_path):
                continue

            # skip empty label files
            with open(label_path, "r") as f:
                lines = f.readlines()
                if len(lines) == 0:
                    continue

            self.images.append(img_name)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]

        img_path = os.path.join(self.img_dir, img_name)
        label_path = os.path.join(
            self.label_dir,
            img_name.replace(".jpg", ".txt").replace(".png", ".txt")
        )

        # load image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # load label safely
        with open(label_path, "r") as f:
            lines = f.readlines()
            label = int(lines[0].split()[0])  # safe now

        if self.transform:
            image = self.transform(image)

        return image, label

# Data Transforms & Loaders

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 128)),   # fast on CPU
    transforms.ToTensor()
])

train_dataset = YOLODataset(
    "/content/nutrivision_merged/train/images",
    "/content/nutrivision_merged/train/labels",
    transform
)

val_dataset = YOLODataset(
    "/content/nutrivision_merged/valid/images",
    "/content/nutrivision_merged/valid/labels",
    transform
)

test_dataset = YOLODataset(
    "/content/nutrivision_merged/test/images",
    "/content/nutrivision_merged/test/labels",
    transform
)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)
print("Train size:", len(train_dataset))

Train size: 29801


# Model & Optimizer Setup

In [ ]:
model = models.mobilenet_v2(weights="DEFAULT")

# detect number of classes dynamically
num_classes = len(set([label for _, label in train_dataset]))

model.classifier[1] = nn.Linear(model.last_channel, num_classes)

model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 98.3MB/s]


# Model Training

In [ ]:
model.train()

for epoch in range(10):
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Epoch 0, Loss: 3686.9720
Epoch 1, Loss: 2275.1470
Epoch 2, Loss: 1759.5053
Epoch 3, Loss: 1469.3736
Epoch 4, Loss: 1224.5268
Epoch 5, Loss: 1091.1631
Epoch 6, Loss: 943.7083
Epoch 7, Loss: 839.5762
Epoch 8, Loss: 751.3158
Epoch 9, Loss: 694.0365


# Model Evaluation

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import torch

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        outputs = model(x)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print(classification_report(all_labels, all_preds))

Accuracy: 0.668993288590604
              precision    recall  f1-score   support

           0       0.60      0.55      0.57        11
           1       0.65      0.59      0.62        61
           2       0.82      0.38      0.52        60
           3       0.35      0.75      0.48         8
           4       0.33      0.50      0.40        10
           5       0.38      0.71      0.50         7
           6       0.88      0.58      0.70        12
           7       0.80      0.75      0.77        16
           8       0.98      0.96      0.97        46
           9       0.60      0.69      0.64        13
          10       1.00      0.50      0.67         4
          11       0.55      0.67      0.60         9
          12       0.12      0.71      0.20         7
          13       1.00      0.94      0.97        52
          14       0.88      0.80      0.84        61
          15       0.95      0.61      0.75        67
          16       0.86      0.30      0.45        63

# Test Evaluation

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import torch

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:   # 👈 use TEST loader
        x = x.to(device)

        outputs = model(x)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())

# ✅ Metrics
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("Precision:", precision_score(all_labels, all_preds, average='macro'))
print("Recall:", recall_score(all_labels, all_preds, average='macro'))
print("F1-score:", f1_score(all_labels, all_preds, average='macro'))

print("\nDetailed Report:\n")
print(classification_report(all_labels, all_preds))

Accuracy: 0.7206119162640902
Precision: 0.7146280578606601
Recall: 0.7042376048553854
F1-score: 0.6883117853119837

Detailed Report:

              precision    recall  f1-score   support

           0       0.67      0.20      0.31        20
           1       0.63      0.52      0.57        52
           2       0.79      0.58      0.67        45
           3       0.60      0.60      0.60        10
           4       0.53      0.75      0.62        12
           5       0.86      0.40      0.55        15
           6       0.75      0.38      0.50         8
           7       0.77      0.67      0.71        15
           8       0.97      0.97      0.97        63
           9       0.50      0.67      0.57        12
          10       1.00      0.44      0.62         9
          11       0.50      0.73      0.59        11
          12       0.39      0.88      0.55        17
          13       1.00      0.98      0.99        58
          14       0.86      0.77      0.81        47
 